# Controleer Lottowinst
In deze notebook moet je de *winstrang* berekenen voor de lottotrekkingen. De winstrang wordt gebruikt om te bepalen hoeveel je gewonnen hebt. In welke winstrang je valt, hangt af van hoeveel juiste cijfers je hebt aangekruist en of je eventueel het bonusnr had aangekruist.
## De winstrangen
De winst van de Lotto wordt berekend op basis van de 'rang':
```
winstrang    aantal winnende nummers     bonusnr
1                                  6           /
2                                  5           1
3                                  5           0
4                                  4           1
5                                  4           0
6                                  3           1
7                                  3           0
8                                  2           1
9                                  1           1
```
Stel dat de volgende getallen zijn getrokken: 8, 9, 13, 20, 21, 22 en bonusnummer = 16

En jij hebt de volgende nummers aangekruist: 8, 9, 13, 20, 21, 15. Dan heb je 5 getallen juist, maar niet het bonusnummer. Je winstrang is dus 3.

Maar stel dat je de volgende nummers hebt aangekruist: 8, 9, 13, 20, 21, 16. Dan heb je ook 5 getallen juist, maar ook het bonusnummer. Daar is je winstrang 2.

De bedoeling van deze oefening is dat je
1. de winstrang berekent voor elke trekking in het bestand dat we hier downloaden.
1. die code gebruikt om een testfunctie te maken: check_frame(df, getallen) (zie einde van deze notebook). Die functie moet gebruikt kunnen worden om te testen je code werkt
We beginnen met de data te downloaden en in te lezen

In [1]:
from pathlib import Path
import requests

STATISTIEKEN = 'data/statistieken-lotto-12-25.xlsx'
statistieken_path = Path(STATISTIEKEN)
if not statistieken_path.exists():
    URL='https://www.nationale-loterij.be/content/dam/opp/draw-games/lotto/brand-assets/documents/nl/statistieken-lotto-12-25.xlsx'
    headers = {'User-Agent': 'Syntra browser'}
    data = requests.get(URL, headers=headers)

    with open(STATISTIEKEN, mode='wb') as f:
        f.write(data.content)
else:
    print('Bestand moet niet gedownload worden.')

Bestand moet niet gedownload worden.


## lees de gegevens in Pandas

In [ ]:
import pandas as pd

STATISTIEKEN = 'data/statistieken-lotto-12-25.xlsx'
df_orig = pd.read_excel(STATISTIEKEN, sheet_name='Resultaten', engine='openpyxl', skiprows=4, nrows=1488, index_col='Trekkingsdatum')
df_orig.info()
df_orig.head(5)

<class 'pandas.DataFrame'>
DatetimeIndex: 1488 entries, 2025-12-31 to 2011-10-01
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   N°1       1488 non-null   int64
 1   N°2       1488 non-null   int64
 2   N°3       1488 non-null   int64
 3   N°4       1488 non-null   int64
 4   N°5       1488 non-null   int64
 5   N°6       1488 non-null   int64
 6   Bonusnr.  1488 non-null   int64
dtypes: int64(7)
memory usage: 93.0 KB


,N°1,N°2,N°3,N°4,N°5,N°6,Bonusnr.
Trekkingsdatum,,,,,,,
2025-12-31,8,9,13,20,21,29,16
2025-12-27,1,25,29,33,38,41,30
2025-12-24,1,5,9,12,15,44,14
2025-12-20,10,11,14,21,37,42,33
2025-12-17,13,16,20,27,41,43,28


# Schrijf je code om bij elke record in df de winstrang te zetten

NB. De check_frame() functie hieronder geeft de winstrang van _één_ record van een df de winstrang terug. De code die bij _elke_ record in een df de winstrang zet staat daarna in de bereken_rangs() functie.

Eerst komt er een hulpfunctie die de mapping doet tussen de rangs en de winnende combinatie van aantal juiste getallen met het bonusnummer.

In [16]:
import numpy as np

# een dictionary met de winnende combinaties en toebehorende rangs:
winnende_combinaties = ['60', '51', '50', '41', '40', '31', '30', '21', '11']
rangs = np.arange(1, 10)
winstrangs = dict(zip(winnende_combinaties, rangs))

# Hulpfunctie
def convert_to_rang(combinatie, winstrangs=winstrangs):
    """ Deze functie zet de winnende combinatie om naar een rang """
    
    if combinatie in winstrangs.keys():
        return winstrangs[combinatie]
    else:
        return 0

## De code hieronder berekent de winstrang voor 1 rij van een DataFrame
We maken een array met winstrangen voor de rangen 0 (niets gewonnen) tot 9 en een array met extra testen voor 'niets gewonnen'. De functie *check_frame* moet de winstrang berekenen.

In de hoofdcode nemen we de eerste rij van het dataframe dat we hebben ingelezen (als een dataframe). We gebruiken dat frame om de functie te testen.

In [6]:
def check_frame(frame, getallen):
    """
    De functie berekent de winstrang voor een reeks getallen
    
    Args:
        frame (pandas.core.frame.DataFrame): Een dataframe van 1 rij met het trekkingsresultaat
        getallen (Iterable): de reeks getallen die gecontroleerd moet worden

    Returns:
        int: De winstrang van de getallen voor dat trekkingsresultaat
    """

    # converteer trekking naar index om intersectie te gebruiken
    trekking = frame.loc[:, frame.columns.str.startswith('N')].to_numpy()
    trekking = pd.Index(trekking.flatten())
    aantal_juist = len(trekking.intersection(getallen))

    # combineer het aantal juiste getallen met een bonus
    bonus_nr = frame.loc[frame.index[0],'Bonusnr.']
    is_bonus = 0
    if bonus_nr in getallen:
        is_bonus = 1
    # winnende combinatie als string
    combinatie = f"{aantal_juist}{is_bonus}"

    return convert_to_rang(combinatie)


getallen = np.array([[1, 2, 3, 4, 5, 6],    # niets gewonnen (rang 0)
                    [8, 9, 13, 20, 21, 29], # rang 1
                    [8, 9, 13, 20, 21, 16], # rang 2
                    [8, 9, 13, 20, 21, 15], # rang 3
                    [8, 9, 13, 20, 25, 16], # rang 4
                    [8, 9, 13, 20, 25, 15], # rang 5
                    [8, 9, 13, 24, 25, 16], # rang 6
                    [8, 9, 13, 24, 25, 15], # rang 7
                    [8, 9, 23, 24, 25, 16], # rang 8
                    [8, 22, 23, 24, 25, 16]]) # rang 9

# extra controles op 'niet gewonnen'
getallen_rang0 = np.array([[8, 2, 3, 4, 5, 6],   # rang 0 (1 getal juist zonder bonus)
                           [8, 9, 3, 4, 5, 6],   # rang 0 (2 getallen juist zonder bonus)
                           [1, 2, 3, 4, 5, 16]]) # rang 0 (alleen bonus juist)
controle_frame = df_orig.iloc[:1]  # Gebruik de eerste rij van het dataframe (getallen: 8, 9, 13, 20, 21, 29)

for rang, rij in enumerate(getallen):
    berekende_rang = check_frame(controle_frame, rij)
    if berekende_rang == rang:
        print(f'Rang {rang} is ok.')
    else:
        print(f'Verwacht: {rang}, resultaat: {berekende_rang}')

print("Extra controle op 'niet gewonnen'")
for rij in getallen_rang0:
    # (hier stond "df", ik heb die naar controle_frame aangepast volgens de "Args" hierboven):
    berekende_rang = check_frame(controle_frame, rij)
    if berekende_rang == 0:
        print(f'controle {rij} is ok.' )
    else:
        print(f'controle op {rij} niet ok: gevonden rang is {berekende_rang}')

Rang 0 is ok.
Rang 1 is ok.
Rang 2 is ok.
Rang 3 is ok.
Rang 4 is ok.
Rang 5 is ok.
Rang 6 is ok.
Rang 7 is ok.
Rang 8 is ok.
Rang 9 is ok.
Extra controle op 'niet gewonnen'
controle [8 2 3 4 5 6] is ok.
controle [8 9 3 4 5 6] is ok.
controle [ 1  2  3  4  5 16] is ok.


### De functie bereken_rangs hieronder voegt winstrangs als extra kolom aan de DataFrame:

In [15]:
def bereken_rangs(frame, getallen):
    """
    Deze functie berekent de winstrang voor een gegeven getal bij elke trekking
    en voegt die als een extra kolom aan de DataFrame 
    """

    df = frame.copy()
    
    # tijdelijke kolommen voor elk aangekruiste getal
    for getal in getallen:
        df[f'getal{getal}'] = df.loc[:, df.columns.str.startswith('N')].eq(getal).any(axis='columns')

    # toevoegen de som, al dan niet bonus en de combinatie daarvan
    df['aantal_juist'] = df.loc[:, df.columns.str.startswith('getal')].sum(axis='columns')
    df['is_bonus'] = df.loc[:, 'Bonusnr.'].isin(getallen).astype(int)
    df['comb'] = df['aantal_juist'].astype(str) + df['is_bonus'].astype(str)

    # mapping combinatie met de rang
    df['winstrang'] = list(map(convert_to_rang, df['comb']))

    # verwijder tijdelijke kolommen
    df_met_winstrangs = df.drop(df.loc[:, df.columns.str.startswith('getal')], axis=1).drop('comb', axis=1)

    return df_met_winstrangs


# 6 unieke integers tussen 1 en 45
rng = np.random.default_rng()
aangekruist = rng.choice(np.arange(1, 46), size=6, replace=False)

# voeg een kolom met rangs toe
df_met_rangs = bereken_rangs(df_orig, aangekruist)

# mogelijke output
all_rangs = df_met_rangs[df_met_rangs['winstrang'] != 0].sort_values('winstrang')
print(f"De nummers {aangekruist} zouden {all_rangs.shape[0]} keer een winstrang hebben (de hoogste: {all_rangs.iat[0,-1]})")
display(all_rangs.head(7))


De nummers [40 26 13 28 25 29] zouden 127 keer een winstrang hebben (de hoogste: 4)


,N°1,N°2,N°3,N°4,N°5,N°6,Bonusnr.,aantal_juist,is_bonus,winstrang
Trekkingsdatum,,,,,,,,,,
2013-05-01,1,13,26,28,36,40,29,4,1,4
2015-01-17,13,24,25,26,28,44,18,4,0,5
2024-09-21,24,25,27,28,34,40,26,3,1,6
2019-02-23,3,25,28,32,40,43,29,3,1,6
2016-04-09,11,13,21,22,25,26,40,3,1,6
2025-04-12,17,21,25,28,29,35,10,3,0,7
2025-07-30,9,13,29,32,35,40,16,3,0,7
